# O2O优惠券核销预测——特征工程

基于前期数据清洗与探索性分析结果，从用户、商户、优惠券、距离及领券时间等维度构建预测特征，为后续优惠券15天内核销预测模型提供输入数据。

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")

coupon_data = pd.read_parquet(
    DATA_DIR / "coupon_basic.parquet"
)

print("数据读取完成")
print("数据形状：", coupon_data.shape)

coupon_data.head()

数据读取完成
数据形状： (1053282, 19)


,User_id,Merchant_id,Coupon_id,Discount_rate,Distance,Date_received,Date,days_to_use,label,is_manjian,discount_rate_value,discount_threshold,discount_reduction,distance_missing,distance_value,receive_month,receive_day,receive_weekday,is_weekend
0,1439408,4663,11002.0,150:20,1.0,2016-05-28,NaT,NaN,0,1,0.866667,150.0,20.0,0,1.0,5,28,5,1
1,1439408,2632,8591.0,20:1,0.0,2016-02-17,NaT,NaN,0,1,0.950000,20.0,1.0,0,0.0,2,17,2,0
2,1439408,2632,1078.0,20:1,0.0,2016-03-19,NaT,NaN,0,1,0.950000,20.0,1.0,0,0.0,3,19,5,1
3,1439408,2632,8591.0,20:1,0.0,2016-06-13,NaT,NaN,0,1,0.950000,20.0,1.0,0,0.0,6,13,0,0
4,1439408,2632,8591.0,20:1,0.0,2016-05-16,2016-06-13,28.0,0,1,0.950000,20.0,1.0,0,0.0,5,16,0,0


## 1. 基础特征构建

基于用户领券时已知的信息构建优惠券、距离和时间特征，并排除实际核销日期、核销间隔等可能造成数据泄漏的字段。

In [2]:
# 构建基础特征数据
feature_data = coupon_data.copy()

# 当前可直接用于建模的基础特征
basic_features = [
    "is_manjian",
    "discount_rate_value",
    "discount_threshold",
    "discount_reduction",
    "distance_missing",
    "distance_value",
    "receive_month",
    "receive_day",
    "receive_weekday",
    "is_weekend"
]

print("基础特征数量：", len(basic_features))
print("\n基础特征：")

for feature in basic_features:
    print("-", feature)

feature_data[basic_features + ["label"]].head()

基础特征数量： 10

基础特征：
- is_manjian
- discount_rate_value
- discount_threshold
- discount_reduction
- distance_missing
- distance_value
- receive_month
- receive_day
- receive_weekday
- is_weekend


,is_manjian,discount_rate_value,discount_threshold,discount_reduction,distance_missing,distance_value,receive_month,receive_day,receive_weekday,is_weekend,label
0,1,0.866667,150.0,20.0,0,1.0,5,28,5,1,0
1,1,0.950000,20.0,1.0,0,0.0,2,17,2,0,0
2,1,0.950000,20.0,1.0,0,0.0,3,19,5,1,0
3,1,0.950000,20.0,1.0,0,0.0,6,13,0,0,0
4,1,0.950000,20.0,1.0,0,0.0,5,16,0,0,0


In [3]:
# 检查基础特征缺失情况
feature_missing = (
    feature_data[basic_features]
    .isnull()
    .sum()
    .to_frame("missing_count")
)

feature_missing["missing_rate(%)"] = (
    feature_missing["missing_count"]
    / len(feature_data)
    * 100
).round(2)

feature_missing

,missing_count,missing_rate(%)
is_manjian,0,0.0
discount_rate_value,0,0.0
discount_threshold,0,0.0
discount_reduction,0,0.0
distance_missing,0,0.0
distance_value,0,0.0
receive_month,0,0.0
receive_day,0,0.0
receive_weekday,0,0.0
is_weekend,0,0.0


## 2. 用户历史行为特征

按照领券时间顺序构建用户历史行为特征，仅使用当前领券记录之前的信息，避免未来数据泄漏。主要包括用户历史领券次数、历史核销次数及历史核销率。

In [4]:
# 按用户和领券时间排序
feature_data = feature_data.sort_values(
    ["User_id", "Date_received"]
).reset_index(drop=True)

feature_data[
    ["User_id", "Date_received", "label"]
].head(20)

,User_id,Date_received,label
0,4,2016-02-14,0
1,4,2016-06-07,0
2,35,2016-01-29,0
3,35,2016-01-29,0
4,35,2016-01-30,0
5,35,2016-01-30,0
6,36,2016-01-25,0
7,36,2016-01-25,0
8,64,2016-01-29,0
9,110,2016-01-31,0


In [5]:
# 用户在当前记录之前的历史领券次数
feature_data["user_hist_coupon_count"] = (
    feature_data
    .groupby("User_id")
    .cumcount()
)

feature_data[
    [
        "User_id",
        "Date_received",
        "user_hist_coupon_count"
    ]
].head(20)

,User_id,Date_received,user_hist_coupon_count
0,4,2016-02-14,0
1,4,2016-06-07,1
2,35,2016-01-29,0
3,35,2016-01-29,1
4,35,2016-01-30,2
5,35,2016-01-30,3
6,36,2016-01-25,0
7,36,2016-01-25,1
8,64,2016-01-29,0
9,110,2016-01-31,0


In [6]:
# 用户在当前记录之前的历史核销次数
feature_data["user_hist_redeem_count"] = (
    feature_data
    .groupby("User_id")["label"]
    .cumsum()
    - feature_data["label"]
)

feature_data[
    [
        "User_id",
        "Date_received",
        "label",
        "user_hist_coupon_count",
        "user_hist_redeem_count"
    ]
].head(20)

,User_id,Date_received,label,user_hist_coupon_count,user_hist_redeem_count
0,4,2016-02-14,0,0,0
1,4,2016-06-07,0,1,0
2,35,2016-01-29,0,0,0
3,35,2016-01-29,0,1,0
4,35,2016-01-30,0,2,0
5,35,2016-01-30,0,3,0
6,36,2016-01-25,0,0,0
7,36,2016-01-25,0,1,0
8,64,2016-01-29,0,0,0
9,110,2016-01-31,0,0,0


In [7]:
# 用户历史核销率
feature_data["user_hist_redeem_rate"] = np.where(
    feature_data["user_hist_coupon_count"] > 0,
    feature_data["user_hist_redeem_count"]
    / feature_data["user_hist_coupon_count"],
    0
)

feature_data[
    [
        "User_id",
        "Date_received",
        "label",
        "user_hist_coupon_count",
        "user_hist_redeem_count",
        "user_hist_redeem_rate"
    ]
].head(30)

,User_id,Date_received,label,user_hist_coupon_count,user_hist_redeem_count,user_hist_redeem_rate
0,4,2016-02-14,0,0,0,0.0
1,4,2016-06-07,0,1,0,0.0
2,35,2016-01-29,0,0,0,0.0
3,35,2016-01-29,0,1,0,0.0
4,35,2016-01-30,0,2,0,0.0
5,35,2016-01-30,0,3,0,0.0
6,36,2016-01-25,0,0,0,0.0
7,36,2016-01-25,0,1,0,0.0
8,64,2016-01-29,0,0,0,0.0
9,110,2016-01-31,0,0,0,0.0


In [8]:
# 找一个既有核销记录、又有多次领券记录的用户进行检查
check_user = (
    feature_data
    .groupby("User_id")
    .agg(
        record_count=("label", "size"),
        redeem_count=("label", "sum")
    )
    .query("record_count >= 5 and redeem_count >= 2")
    .index[0]
)

print("检查用户ID：", check_user)

feature_data.loc[
    feature_data["User_id"] == check_user,
    [
        "User_id",
        "Date_received",
        "label",
        "user_hist_coupon_count",
        "user_hist_redeem_count",
        "user_hist_redeem_rate"
    ]
]

检查用户ID： 687


,User_id,Date_received,label,user_hist_coupon_count,user_hist_redeem_count,user_hist_redeem_rate
90,687,2016-01-28,0,0,0,0.000000
91,687,2016-01-28,0,1,0,0.000000
92,687,2016-01-28,1,2,0,0.000000
93,687,2016-01-28,0,3,1,0.333333
94,687,2016-01-30,0,4,1,0.250000
95,687,2016-01-30,0,5,1,0.200000
96,687,2016-01-30,0,6,1,0.166667
97,687,2016-03-28,1,7,1,0.142857


## 3. 商户历史行为特征

按照领券时间构建商户历史优惠券发放与核销特征，用于刻画不同商户优惠券的历史吸引力和核销表现。

In [9]:
# 按商户和领券时间排序
feature_data = feature_data.sort_values(
    ["Merchant_id", "Date_received"]
).reset_index(drop=True)

# 当前记录之前，该商户历史发券次数
feature_data["merchant_hist_coupon_count"] = (
    feature_data
    .groupby("Merchant_id")
    .cumcount()
)

feature_data[
    [
        "Merchant_id",
        "Date_received",
        "merchant_hist_coupon_count"
    ]
].head(20)

,Merchant_id,Date_received,merchant_hist_coupon_count
0,2,2016-05-15,0
1,2,2016-05-16,1
2,2,2016-05-17,2
3,2,2016-05-18,3
4,2,2016-05-19,4
5,2,2016-05-20,5
6,2,2016-05-20,6
7,3,2016-05-25,0
8,3,2016-05-28,1
9,3,2016-05-28,2


In [10]:
# 当前记录之前，该商户历史核销次数
feature_data["merchant_hist_redeem_count"] = (
    feature_data
    .groupby("Merchant_id")["label"]
    .cumsum()
    - feature_data["label"]
)

# 商户历史核销率
feature_data["merchant_hist_redeem_rate"] = np.where(
    feature_data["merchant_hist_coupon_count"] > 0,
    feature_data["merchant_hist_redeem_count"]
    / feature_data["merchant_hist_coupon_count"],
    0
)

feature_data[
    [
        "Merchant_id",
        "Date_received",
        "label",
        "merchant_hist_coupon_count",
        "merchant_hist_redeem_count",
        "merchant_hist_redeem_rate"
    ]
].head(20)

,Merchant_id,Date_received,label,merchant_hist_coupon_count,merchant_hist_redeem_count,merchant_hist_redeem_rate
0,2,2016-05-15,0,0,0,0.0
1,2,2016-05-16,0,1,0,0.0
2,2,2016-05-17,0,2,0,0.0
3,2,2016-05-18,0,3,0,0.0
4,2,2016-05-19,0,4,0,0.0
5,2,2016-05-20,0,5,0,0.0
6,2,2016-05-20,0,6,0,0.0
7,3,2016-05-25,0,0,0,0.0
8,3,2016-05-28,0,1,0,0.0
9,3,2016-05-28,0,2,0,0.0


In [11]:
# 找一个记录较多且存在核销行为的商户进行检查
check_merchant = (
    feature_data
    .groupby("Merchant_id")
    .agg(
        record_count=("label", "size"),
        redeem_count=("label", "sum")
    )
    .query("record_count >= 10 and redeem_count >= 2")
    .index[0]
)

print("检查商户ID：", check_merchant)

feature_data.loc[
    feature_data["Merchant_id"] == check_merchant,
    [
        "Merchant_id",
        "Date_received",
        "label",
        "merchant_hist_coupon_count",
        "merchant_hist_redeem_count",
        "merchant_hist_redeem_rate"
    ]
].head(30)

检查商户ID： 5


,Merchant_id,Date_received,label,merchant_hist_coupon_count,merchant_hist_redeem_count,merchant_hist_redeem_rate
24,5,2016-05-12,0,0,0,0.000000
25,5,2016-05-14,0,1,0,0.000000
26,5,2016-05-15,0,2,0,0.000000
27,5,2016-05-15,0,3,0,0.000000
28,5,2016-05-15,0,4,0,0.000000
29,5,2016-05-16,0,5,0,0.000000
30,5,2016-05-18,0,6,0,0.000000
31,5,2016-05-18,0,7,0,0.000000
32,5,2016-05-19,0,8,0,0.000000
33,5,2016-05-19,0,9,0,0.000000


## 4. 优惠券历史特征

按照领券时间构建优惠券历史领取次数、历史核销次数和历史核销率，用于刻画不同优惠券方案过去的实际使用表现。

In [12]:
# 按优惠券和领券时间排序
feature_data = feature_data.sort_values(
    ["Coupon_id", "Date_received"]
).reset_index(drop=True)

# 当前记录之前，该优惠券历史被领取次数
feature_data["coupon_hist_receive_count"] = (
    feature_data
    .groupby("Coupon_id")
    .cumcount()
)

feature_data[
    [
        "Coupon_id",
        "Date_received",
        "coupon_hist_receive_count"
    ]
].head(20)

,Coupon_id,Date_received,coupon_hist_receive_count
0,1.0,2016-05-13,0
1,1.0,2016-05-22,1
2,1.0,2016-05-30,2
3,1.0,2016-06-03,3
4,1.0,2016-06-06,4
5,2.0,2016-05-10,0
6,2.0,2016-05-12,1
7,3.0,2016-05-04,0
8,3.0,2016-05-07,1
9,3.0,2016-05-13,2


In [13]:
# 当前记录之前，该优惠券历史核销次数
feature_data["coupon_hist_redeem_count"] = (
    feature_data
    .groupby("Coupon_id")["label"]
    .cumsum()
    - feature_data["label"]
)

# 优惠券历史核销率
feature_data["coupon_hist_redeem_rate"] = np.where(
    feature_data["coupon_hist_receive_count"] > 0,
    feature_data["coupon_hist_redeem_count"]
    / feature_data["coupon_hist_receive_count"],
    0
)

feature_data[
    [
        "Coupon_id",
        "Date_received",
        "label",
        "coupon_hist_receive_count",
        "coupon_hist_redeem_count",
        "coupon_hist_redeem_rate"
    ]
].head(20)

,Coupon_id,Date_received,label,coupon_hist_receive_count,coupon_hist_redeem_count,coupon_hist_redeem_rate
0,1.0,2016-05-13,0,0,0,0.000000
1,1.0,2016-05-22,1,1,0,0.000000
2,1.0,2016-05-30,0,2,1,0.500000
3,1.0,2016-06-03,0,3,1,0.333333
4,1.0,2016-06-06,0,4,1,0.250000
5,2.0,2016-05-10,1,0,0,0.000000
6,2.0,2016-05-12,0,1,1,1.000000
7,3.0,2016-05-04,0,0,0,0.000000
8,3.0,2016-05-07,0,1,0,0.000000
9,3.0,2016-05-13,0,2,0,0.000000


In [14]:
# 找一张记录较多且存在核销行为的优惠券进行检查
check_coupon = (
    feature_data
    .groupby("Coupon_id")
    .agg(
        record_count=("label", "size"),
        redeem_count=("label", "sum")
    )
    .query("record_count >= 10 and redeem_count >= 2")
    .index[0]
)

print("检查优惠券ID：", check_coupon)

feature_data.loc[
    feature_data["Coupon_id"] == check_coupon,
    [
        "Coupon_id",
        "Date_received",
        "label",
        "coupon_hist_receive_count",
        "coupon_hist_redeem_count",
        "coupon_hist_redeem_rate"
    ]
].head(30)

检查优惠券ID： 4.0


,Coupon_id,Date_received,label,coupon_hist_receive_count,coupon_hist_redeem_count,coupon_hist_redeem_rate
22,4.0,2016-01-03,0,0,0,0.000000
23,4.0,2016-01-08,0,1,0,0.000000
24,4.0,2016-01-15,0,2,0,0.000000
25,4.0,2016-02-10,0,3,0,0.000000
26,4.0,2016-02-24,0,4,0,0.000000
27,4.0,2016-02-25,0,5,0,0.000000
28,4.0,2016-03-05,0,6,0,0.000000
29,4.0,2016-03-05,0,7,0,0.000000
30,4.0,2016-03-09,0,8,0,0.000000
31,4.0,2016-03-09,0,9,0,0.000000


## 5. 时间范围与建模窗口检查

在构建最终历史行为特征前，检查样本的领券时间分布，为后续按照时间窗口划分训练集、验证集及历史统计区间提供依据。

In [15]:
# 查看领券日期范围
print("最早领券日期：", feature_data["Date_received"].min())
print("最晚领券日期：", feature_data["Date_received"].max())
print("领券天数：", feature_data["Date_received"].nunique())

最早领券日期： 2016-01-01 00:00:00
最晚领券日期： 2016-06-15 00:00:00
领券天数： 167


In [16]:
# 按月份查看领券样本和正样本分布
monthly_summary = (
    feature_data
    .assign(
        month=feature_data["Date_received"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        coupon_count=("label", "size"),
        redeem_count=("label", "sum"),
        redeem_rate=("label", "mean")
    )
)

monthly_summary["redeem_rate"] *= 100

monthly_summary.round(2)

,coupon_count,redeem_count,redeem_rate
month,,,
2016-01,371539,9849,2.65
2016-02,133433,5331,4.00
2016-03,103903,15291,14.72
2016-04,138094,5833,4.22
2016-05,215348,21580,10.02
2016-06,90965,6511,7.16


## 6. 构建无泄漏历史行为特征

为避免使用预测时点之后才能确定的信息，历史核销特征仅使用在当前领券日期15天以前已经领取的优惠券记录，从而保证历史标签在预测时已经完整成熟。

In [17]:
# 每条优惠券标签完全成熟的日期
feature_data["label_available_date"] = (
    feature_data["Date_received"]
    + pd.Timedelta(days=15)
)

feature_data[
    [
        "Date_received",
        "label_available_date",
        "label"
    ]
].head(10)

,Date_received,label_available_date,label
0,2016-05-13,2016-05-28,0
1,2016-05-22,2016-06-06,1
2,2016-05-30,2016-06-14,0
3,2016-06-03,2016-06-18,0
4,2016-06-06,2016-06-21,0
5,2016-05-10,2016-05-25,1
6,2016-05-12,2016-05-27,0
7,2016-05-04,2016-05-19,0
8,2016-05-07,2016-05-22,0
9,2016-05-13,2016-05-28,0


In [18]:
# 按时间划分训练集和验证集
train_mask = (
    feature_data["Date_received"]
    <= "2016-04-30"
)

valid_mask = (
    (feature_data["Date_received"] >= "2016-05-01")
    &
    (feature_data["Date_received"] <= "2016-05-31")
)

print("训练样本数：", train_mask.sum())
print("验证样本数：", valid_mask.sum())

print(
    "训练集日期：",
    feature_data.loc[train_mask, "Date_received"].min(),
    "至",
    feature_data.loc[train_mask, "Date_received"].max()
)

print(
    "验证集日期：",
    feature_data.loc[valid_mask, "Date_received"].min(),
    "至",
    feature_data.loc[valid_mask, "Date_received"].max()
)

训练样本数： 746969
验证样本数： 215348
训练集日期： 2016-01-01 00:00:00 至 2016-04-30 00:00:00
验证集日期： 2016-05-01 00:00:00 至 2016-05-31 00:00:00


## 7. 无泄漏用户历史特征

基于标签成熟日期构建用户历史行为特征。对于当前领券记录，仅统计在当前日期之前已经完成15天观察窗口的历史优惠券，从而避免使用预测时点尚不可知的未来信息。

In [19]:
# 保存每条记录原始位置，方便计算后恢复顺序
feature_data = feature_data.reset_index(drop=True)
feature_data["row_id"] = np.arange(len(feature_data))

# 当前需要预测的记录
current = feature_data[
    ["row_id", "User_id", "Date_received"]
].copy()

current = current.sort_values(
    ["Date_received", "User_id"]
)

# 已经成熟、可以作为历史信息的记录
history = feature_data[
    ["User_id", "label_available_date", "label"]
].copy()

history = history.sort_values(
    ["label_available_date", "User_id"]
)

# 对历史记录做累计统计
history["user_mature_coupon_count"] = (
    history.groupby("User_id").cumcount() + 1
)

history["user_mature_redeem_count"] = (
    history.groupby("User_id")["label"].cumsum()
)

In [24]:
# 按时间向后匹配最近一个已经成熟的历史状态
user_history_features = pd.merge_asof(
    current,
    history,
    left_on="Date_received",
    right_on="label_available_date",
    by="User_id",
    direction="backward",
    allow_exact_matches=False
)

user_history_features[
    [
        "User_id",
        "Date_received",
        "label_available_date",
        "user_mature_coupon_count",
        "user_mature_redeem_count"
    ]
].head(20)

,User_id,Date_received,label_available_date,user_mature_coupon_count,user_mature_redeem_count
0,2135,2016-01-01,NaT,NaN,NaN
1,2550,2016-01-01,NaT,NaN,NaN
2,2981,2016-01-01,NaT,NaN,NaN
3,40922,2016-01-01,NaT,NaN,NaN
4,58705,2016-01-01,NaT,NaN,NaN
5,73115,2016-01-01,NaT,NaN,NaN
6,86171,2016-01-01,NaT,NaN,NaN
7,88189,2016-01-01,NaT,NaN,NaN
8,90059,2016-01-01,NaT,NaN,NaN
9,90980,2016-01-01,NaT,NaN,NaN


In [28]:
feature_cols = [
    "user_mature_coupon_count",
    "user_mature_redeem_count"
]

user_history_features[feature_cols] = (
    user_history_features[feature_cols]
    .fillna(0)
)

user_history_features["user_mature_redeem_rate"] = np.where(
    user_history_features["user_mature_coupon_count"] > 0,
    user_history_features["user_mature_redeem_count"]
    / user_history_features["user_mature_coupon_count"],
    0
)

In [29]:
# 找一个拥有多条成熟历史记录的用户进行验证
check_row = (
    user_history_features[
        user_history_features["user_mature_coupon_count"] >= 3
    ]
    .iloc[0]
)

check_user = check_row["User_id"]
check_date = check_row["Date_received"]

print("检查用户：", check_user)
print("当前领券日期：", check_date)
print("当前可用成熟历史次数：", check_row["user_mature_coupon_count"])
print("当前可用成熟历史核销次数：", check_row["user_mature_redeem_count"])
print("当前成熟历史核销率：", check_row["user_mature_redeem_rate"])

print("\n实际允许使用的历史记录：")

feature_data.loc[
    (feature_data["User_id"] == check_user)
    & (feature_data["label_available_date"] < check_date),
    [
        "User_id",
        "Date_received",
        "label_available_date",
        "label"
    ]
].sort_values("Date_received")

检查用户： 915955
当前领券日期： 2016-01-20 00:00:00
当前可用成熟历史次数： 3.0
当前可用成熟历史核销次数： 1.0
当前成熟历史核销率： 0.3333333333333333

实际允许使用的历史记录：


,User_id,Date_received,label_available_date,label
680365,915955,2016-01-02,2016-01-17,1
680366,915955,2016-01-03,2016-01-18,0
680368,915955,2016-01-04,2016-01-19,0


## 8. 无泄漏商户与优惠券历史特征

采用与用户历史特征相同的标签成熟机制，分别构建商户和优惠券维度的历史领取、历史核销及历史核销率特征。

In [31]:
# =========================
# 无泄漏商户历史特征
# =========================

current_merchant = feature_data[
    ["row_id", "Merchant_id", "Date_received"]
].copy()

current_merchant = current_merchant.sort_values(
    ["Date_received", "Merchant_id"]
)

merchant_history = feature_data[
    ["Merchant_id", "label_available_date", "label"]
].copy()

merchant_history = merchant_history.sort_values(
    ["label_available_date", "Merchant_id"]
)

# 商户成熟历史累计统计
merchant_history["merchant_mature_coupon_count"] = (
    merchant_history
    .groupby("Merchant_id")
    .cumcount() + 1
)

merchant_history["merchant_mature_redeem_count"] = (
    merchant_history
    .groupby("Merchant_id")["label"]
    .cumsum()
)

# 匹配当前日期之前已经成熟的历史状态
merchant_history_features = pd.merge_asof(
    current_merchant,
    merchant_history,
    left_on="Date_received",
    right_on="label_available_date",
    by="Merchant_id",
    direction="backward",
    allow_exact_matches=False
)

# 无成熟历史时填0
merchant_cols = [
    "merchant_mature_coupon_count",
    "merchant_mature_redeem_count"
]

merchant_history_features[merchant_cols] = (
    merchant_history_features[merchant_cols]
    .fillna(0)
)

# 商户成熟历史核销率
merchant_history_features["merchant_mature_redeem_rate"] = np.where(
    merchant_history_features["merchant_mature_coupon_count"] > 0,
    merchant_history_features["merchant_mature_redeem_count"]
    / merchant_history_features["merchant_mature_coupon_count"],
    0
)

merchant_history_features[
    [
        "Merchant_id",
        "Date_received",
        "merchant_mature_coupon_count",
        "merchant_mature_redeem_count",
        "merchant_mature_redeem_rate"
    ]
].tail(20)

,Merchant_id,Date_received,merchant_mature_coupon_count,merchant_mature_redeem_count,merchant_mature_redeem_rate
1053262,8761,2016-06-15,7.0,0.0,0.000000
1053263,8761,2016-06-15,7.0,0.0,0.000000
1053264,8766,2016-06-15,5.0,0.0,0.000000
1053265,8773,2016-06-15,56.0,11.0,0.196429
1053266,8773,2016-06-15,56.0,11.0,0.196429
1053267,8776,2016-06-15,0.0,0.0,0.000000
1053268,8796,2016-06-15,2.0,2.0,1.000000
1053269,8808,2016-06-15,24.0,6.0,0.250000
1053270,8808,2016-06-15,24.0,6.0,0.250000
1053271,8808,2016-06-15,24.0,6.0,0.250000


In [32]:
# =========================
# 无泄漏优惠券历史特征
# =========================

current_coupon = feature_data[
    ["row_id", "Coupon_id", "Date_received"]
].copy()

current_coupon = current_coupon.sort_values(
    ["Date_received", "Coupon_id"]
)

coupon_history = feature_data[
    ["Coupon_id", "label_available_date", "label"]
].copy()

coupon_history = coupon_history.sort_values(
    ["label_available_date", "Coupon_id"]
)

# 优惠券成熟历史累计统计
coupon_history["coupon_mature_receive_count"] = (
    coupon_history
    .groupby("Coupon_id")
    .cumcount() + 1
)

coupon_history["coupon_mature_redeem_count"] = (
    coupon_history
    .groupby("Coupon_id")["label"]
    .cumsum()
)

# 匹配当前日期之前已经成熟的历史状态
coupon_history_features = pd.merge_asof(
    current_coupon,
    coupon_history,
    left_on="Date_received",
    right_on="label_available_date",
    by="Coupon_id",
    direction="backward",
    allow_exact_matches=False
)

coupon_cols = [
    "coupon_mature_receive_count",
    "coupon_mature_redeem_count"
]

coupon_history_features[coupon_cols] = (
    coupon_history_features[coupon_cols]
    .fillna(0)
)

# 优惠券成熟历史核销率
coupon_history_features["coupon_mature_redeem_rate"] = np.where(
    coupon_history_features["coupon_mature_receive_count"] > 0,
    coupon_history_features["coupon_mature_redeem_count"]
    / coupon_history_features["coupon_mature_receive_count"],
    0
)

coupon_history_features[
    [
        "Coupon_id",
        "Date_received",
        "coupon_mature_receive_count",
        "coupon_mature_redeem_count",
        "coupon_mature_redeem_rate"
    ]
].tail(20)

,Coupon_id,Date_received,coupon_mature_receive_count,coupon_mature_redeem_count,coupon_mature_redeem_rate
1053262,13815.0,2016-06-15,0.0,0.0,0.000000
1053263,13817.0,2016-06-15,0.0,0.0,0.000000
1053264,13865.0,2016-06-15,34.0,0.0,0.000000
1053265,13865.0,2016-06-15,34.0,0.0,0.000000
1053266,13884.0,2016-06-15,23.0,5.0,0.217391
1053267,13892.0,2016-06-15,5.0,0.0,0.000000
1053268,13892.0,2016-06-15,5.0,0.0,0.000000
1053269,13908.0,2016-06-15,10.0,4.0,0.400000
1053270,13916.0,2016-06-15,65.0,9.0,0.138462
1053271,13928.0,2016-06-15,0.0,0.0,0.000000


## 9. 最终建模特征整理

将基础特征与用户、商户、优惠券三个维度的无泄漏历史行为特征进行合并，形成最终建模数据集，并对特征完整性和潜在数据泄漏进行检查。

In [33]:
# 用户历史特征
user_feature_cols = [
    "row_id",
    "user_mature_coupon_count",
    "user_mature_redeem_count",
    "user_mature_redeem_rate"
]

# 商户历史特征
merchant_feature_cols = [
    "row_id",
    "merchant_mature_coupon_count",
    "merchant_mature_redeem_count",
    "merchant_mature_redeem_rate"
]

# 优惠券历史特征
coupon_feature_cols = [
    "row_id",
    "coupon_mature_receive_count",
    "coupon_mature_redeem_count",
    "coupon_mature_redeem_rate"
]

print("用户历史特征：3个")
print("商户历史特征：3个")
print("优惠券历史特征：3个")

用户历史特征：3个
商户历史特征：3个
优惠券历史特征：3个


In [34]:
# 合并无泄漏历史特征
model_data = feature_data.copy()

model_data = model_data.merge(
    user_history_features[user_feature_cols],
    on="row_id",
    how="left"
)

model_data = model_data.merge(
    merchant_history_features[merchant_feature_cols],
    on="row_id",
    how="left"
)

model_data = model_data.merge(
    coupon_history_features[coupon_feature_cols],
    on="row_id",
    how="left"
)

print("原始记录数：", len(feature_data))
print("合并后记录数：", len(model_data))
print("合并后字段数：", model_data.shape[1])

原始记录数： 1053282
合并后记录数： 1053282
合并后字段数： 39


In [35]:
# 10个基础特征
basic_features = [
    "is_manjian",
    "discount_rate_value",
    "discount_threshold",
    "discount_reduction",
    "distance_missing",
    "distance_value",
    "receive_month",
    "receive_day",
    "receive_weekday",
    "is_weekend"
]

# 9个无泄漏历史特征
history_features = [
    "user_mature_coupon_count",
    "user_mature_redeem_count",
    "user_mature_redeem_rate",

    "merchant_mature_coupon_count",
    "merchant_mature_redeem_count",
    "merchant_mature_redeem_rate",

    "coupon_mature_receive_count",
    "coupon_mature_redeem_count",
    "coupon_mature_redeem_rate"
]

final_features = basic_features + history_features

print("基础特征数量：", len(basic_features))
print("历史特征数量：", len(history_features))
print("最终候选特征数量：", len(final_features))

final_features

基础特征数量： 10
历史特征数量： 9
最终候选特征数量： 19


['is_manjian',
 'discount_rate_value',
 'discount_threshold',
 'discount_reduction',
 'distance_missing',
 'distance_value',
 'receive_month',
 'receive_day',
 'receive_weekday',
 'is_weekend',
 'user_mature_coupon_count',
 'user_mature_redeem_count',
 'user_mature_redeem_rate',
 'merchant_mature_coupon_count',
 'merchant_mature_redeem_count',
 'merchant_mature_redeem_rate',
 'coupon_mature_receive_count',
 'coupon_mature_redeem_count',
 'coupon_mature_redeem_rate']

In [36]:
# 检查最终19个候选特征的缺失情况
final_missing = (
    model_data[final_features]
    .isnull()
    .sum()
    .to_frame("missing_count")
)

final_missing["missing_rate(%)"] = (
    final_missing["missing_count"]
    / len(model_data)
    * 100
).round(2)

final_missing

,missing_count,missing_rate(%)
is_manjian,0,0.0
discount_rate_value,0,0.0
discount_threshold,0,0.0
discount_reduction,0,0.0
distance_missing,0,0.0
distance_value,0,0.0
receive_month,0,0.0
receive_day,0,0.0
receive_weekday,0,0.0
is_weekend,0,0.0


In [37]:
# 明确禁止进入模型的泄漏字段
leakage_features = [
    "Date",
    "days_to_use",
    "label",
    "label_available_date"
]

print("禁止作为模型输入的字段：")
for col in leakage_features:
    print("-", col)

print(
    "\n泄漏字段是否误入最终特征：",
    set(leakage_features) & set(final_features)
)

禁止作为模型输入的字段：
- Date
- days_to_use
- label
- label_available_date

泄漏字段是否误入最终特征： set()


## 10. 建模数据集构建与保存

按照时间顺序划分训练集与验证集，以2016年1月至4月的领券记录作为训练样本，2016年5月的领券记录作为验证样本。模型输入仅保留19个无泄漏候选特征，并单独保存目标变量。

In [38]:
# 按时间划分训练集和验证集
train_mask = (
    model_data["Date_received"] <= "2016-04-30"
)

valid_mask = (
    (model_data["Date_received"] >= "2016-05-01")
    &
    (model_data["Date_received"] <= "2016-05-31")
)

X_train = model_data.loc[train_mask, final_features].copy()
y_train = model_data.loc[train_mask, "label"].copy()

X_valid = model_data.loc[valid_mask, final_features].copy()
y_valid = model_data.loc[valid_mask, "label"].copy()

print("X_train：", X_train.shape)
print("y_train：", y_train.shape)

print("X_valid：", X_valid.shape)
print("y_valid：", y_valid.shape)

X_train： (746969, 19)
y_train： (746969,)
X_valid： (215348, 19)
y_valid： (215348,)


In [39]:
# 检查训练集与验证集目标变量分布
print("训练集：")
print(y_train.value_counts())
print("正样本率：", f"{y_train.mean():.2%}")

print("\n验证集：")
print(y_valid.value_counts())
print("正样本率：", f"{y_valid.mean():.2%}")

训练集：
label
0    710665
1     36304
Name: count, dtype: int64
正样本率： 4.86%

验证集：
label
0    193768
1     21580
Name: count, dtype: int64
正样本率： 10.02%


In [40]:
# 合并特征和目标变量，便于后续建模读取
train_model = X_train.copy()
train_model["label"] = y_train.values

valid_model = X_valid.copy()
valid_model["label"] = y_valid.values

# 保存建模数据
train_model.to_parquet(
    DATA_DIR / "train_model.parquet",
    index=False
)

valid_model.to_parquet(
    DATA_DIR / "valid_model.parquet",
    index=False
)

print("建模数据保存完成")
print("训练集：", train_model.shape)
print("验证集：", valid_model.shape)

print("\n保存文件：")
print((DATA_DIR / "train_model.parquet").resolve())
print((DATA_DIR / "valid_model.parquet").resolve())

建模数据保存完成
训练集： (746969, 20)
验证集： (215348, 20)

保存文件：
C:\Users\User\OneDrive\大三\项目训练\O2O_Coupon_Analysis\data\processed\train_model.parquet
C:\Users\User\OneDrive\大三\项目训练\O2O_Coupon_Analysis\data\processed\valid_model.parquet
